# 🎵 Spectral Agent — Surgery Notebook
**Workflow:**
1. Install deps & download dataset from Kaggle
2. Back up raw dataset to Google Drive
3. Extract + serialize features (train / val / test) to `.pt` files
4. Fast training from pre-serialized features (big batch, fewer epochs)

## ⬇️ Cell 1 — Install & Download Dataset

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install -q torch torchaudio librosa soundfile scikit-learn tqdm kaggle

import os, sys, subprocess
from pathlib import Path

# ── Kaggle credentials (set before running) ───────────────────────────────────
# Option A: upload kaggle.json to Colab then run:
#   !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# Option B: set env vars directly:
# os.environ['KAGGLE_USERNAME'] = 'your_username'
# os.environ['KAGGLE_KEY']      = 'your_key'

DATA_DIR       = Path('/content/data/asvspoof5')
KAGGLE_DATASET = 'aniket202411001/asvspoof5-flac'

def download_dataset(data_dir: Path, dataset: str):
    if data_dir.exists() and any(data_dir.iterdir()):
        print(f'[✔] Dataset already at {data_dir}')
        return
    data_dir.mkdir(parents=True, exist_ok=True)
    print(f'[↓] Downloading {dataset} ...')
    subprocess.run(
        [sys.executable, '-m', 'kaggle', 'datasets', 'download',
         '-d', dataset, '--unzip', '-p', str(data_dir)],
        check=True
    )
    print('[✔] Download complete.')

download_dataset(DATA_DIR, KAGGLE_DATASET)

# Quick sanity check — list top-level contents
print('\nDataset contents:')
for p in sorted(DATA_DIR.iterdir()):
    print(' ', p.name)


## 💾 Cell 2 — Back Up Raw Dataset to Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_BACKUP = Path('/content/drive/MyDrive/MAD_Project/asvspoof5_raw')
DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)

print(f'[→] Syncing {DATA_DIR} → {DRIVE_BACKUP}')
print('    (This copies only NEW files; already-copied files are skipped.)')
result = subprocess.run(
    ['rsync', '-av', '--ignore-existing',
     str(DATA_DIR) + '/', str(DRIVE_BACKUP) + '/'],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print('[!] rsync error:', result.stderr[-500:])
else:
    print('[✔] Backup complete.')


## 🔬 Cell 3 — Feature Extraction & Serialization (one-time)

In [ ]:
import random, time, warnings
import numpy as np
import torch
import librosa
from tqdm import tqdm

warnings.filterwarnings('ignore', category=UserWarning)

# ── Config ────────────────────────────────────────────────────────────────────
DATA_FACTOR  = 0.6       # fraction of files to use
SR           = 16000
CHUNK_DUR    = 3.0
OVERLAP      = 0.5
CACHE_DIR    = Path('/content/drive/MyDrive/MAD_Project/spectral_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ── Helpers ───────────────────────────────────────────────────────────────────
def load_audio(path, target_sr=SR):
    wav, _ = librosa.load(str(path), sr=target_sr, mono=True)
    peak = np.abs(wav).max()
    return (wav / peak).astype(np.float32) if peak > 0 else wav

def pad_or_trim(wav, length):
    if len(wav) >= length: return wav[:length]
    return np.pad(wav, (0, length - len(wav)))

def chunk_audio(wav, sr=SR, dur=CHUNK_DUR, overlap=OVERLAP):
    cl = int(dur * sr); hop = int((dur - overlap) * sr)
    if len(wav) <= cl: return [pad_or_trim(wav, cl)]
    chunks, s = [], 0
    while s + cl <= len(wav):
        chunks.append(wav[s:s+cl].copy()); s += hop
    tail = wav[s:]
    if len(tail) > cl // 2: chunks.append(pad_or_trim(tail, cl))
    return chunks

def extract_spectral_features(chunk, sr=SR):
    mel  = librosa.power_to_db(librosa.feature.melspectrogram(y=chunk, sr=sr, n_mels=40, n_fft=512, hop_length=160), ref=np.max)
    mfcc = librosa.feature.mfcc(y=chunk, sr=sr, n_mfcc=20, n_fft=512, hop_length=160)
    S    = np.abs(librosa.stft(chunk, n_fft=512, hop_length=160))
    fp   = np.linspace(0, S.shape[0]-1, 72).astype(int)
    fb   = np.zeros((70, S.shape[0]), dtype=np.float32)
    for m in range(1, 71):
        l,c,r = fp[m-1], fp[m], fp[m+1]
        for k in range(l, c+1):
            if c != l: fb[m-1,k] = (k-l)/(c-l)
        for k in range(c, r+1):
            if r != c: fb[m-1,k] = (r-k)/(r-c)
    lfcc = np.cos(np.pi/70*(np.arange(20)[:,None]+0.5)*np.arange(70)[None,:]) @ np.log(fb @ S + 1e-8)
    cqt  = np.abs(librosa.cqt(chunk, sr=sr, hop_length=160, n_bins=84))
    cqcc = np.cos(np.pi/84*(np.arange(20)[:,None]+0.5)*np.arange(84)[None,:]) @ np.log(cqt + 1e-8)
    T    = min(mel.shape[1], mfcc.shape[1], lfcc.shape[1], cqcc.shape[1])
    feat = np.concatenate([mel[:,:T], mfcc[:,:T], lfcc[:,:T], cqcc[:,:T]], axis=0)
    return ((feat - feat.mean(1, keepdims=True)) / (feat.std(1, keepdims=True) + 1e-8)).astype(np.float32)

def resolve_split_dir(root: Path, split: str) -> Path:
    """Find the right subdirectory for train/val/test, regardless of naming."""
    direct = root / split
    if direct.exists(): return direct
    mapping = {'train': 'flac_T', 'val': 'flac_D', 'test': 'flac_E_eval'}
    candidate = root / mapping.get(split, '')
    if candidate.exists(): return candidate
    # fallback: search for any dir containing the split name
    for d in root.iterdir():
        if d.is_dir() and split in d.name.lower(): return d
    return direct  # may not exist; will produce empty lists

def collect_files(root: Path, split: str, factor: float):
    split_dir  = resolve_split_dir(root, split)
    bona_dir   = split_dir / 'bonafide'
    spoof_dir  = split_dir / 'spoof'
    bona_files = list(bona_dir.glob('**/*.flac')) + list(bona_dir.glob('**/*.wav'))  if bona_dir.exists() else []
    spoof_files= list(spoof_dir.glob('**/*.flac')) + list(spoof_dir.glob('**/*.wav')) if spoof_dir.exists() else []
    if bona_files:  bona_files  = random.sample(bona_files,  min(len(bona_files),  max(1, int(len(bona_files)  * factor))))
    if spoof_files: spoof_files = random.sample(spoof_files, min(len(spoof_files), max(1, int(len(spoof_files) * factor))))
    print(f'  [{split}] bona={len(bona_files)}  spoof={len(spoof_files)}')
    return [(f,0) for f in bona_files] + [(f,1) for f in spoof_files]

# ── Serialize ─────────────────────────────────────────────────────────────────
def serialize_split(root: Path, split: str, factor: float, out_dir: Path):
    out_file = out_dir / f'{split}.pt'
    if out_file.exists():
        print(f'  [{split}] Cache already exists at {out_file} — skipping.')
        return
    pairs = collect_files(root, split, factor)
    all_feats, all_labels = [], []
    for path, label in tqdm(pairs, desc=f'Extracting {split}'):
        try:
            wav = load_audio(path)
            for chunk in chunk_audio(wav):
                all_feats.append(extract_spectral_features(chunk))
                all_labels.append(label)
        except Exception as e:
            pass  # skip corrupt files
    print(f'  [{split}] {len(all_feats)} chunks extracted — saving ...')
    torch.save({'feats': all_feats, 'labels': all_labels}, out_file)
    print(f'  [{split}] Saved → {out_file}')

print('Starting feature extraction...')
t0 = time.time()
for split in ['train', 'val', 'test']:
    serialize_split(DATA_DIR, split, DATA_FACTOR, CACHE_DIR)
print(f'\n[✔] All splits serialized in {(time.time()-t0)/60:.1f} min')


## 🚀 Cell 4 — Model Definition + Fast Training from Cache

In [ ]:
import json, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from scipy.optimize import brentq
from scipy.interpolate import interp1d
from sklearn.metrics import roc_curve, auc as sklearn_auc, accuracy_score, f1_score

# ── Config ────────────────────────────────────────────────────────────────────
CACHE_DIR   = Path('/content/drive/MyDrive/MAD_Project/spectral_cache')
RESULTS_DIR = Path('/content/drive/MyDrive/MAD_Project/spectral_results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 128   # 40 GB RAM — go big
EPOCHS     = 15    # fast run
LR         = 1e-3
PATIENCE   = 4
NUM_WORKERS= 4

# ── Dataset from cache ────────────────────────────────────────────────────────
class CachedSpectralDataset(Dataset):
    def __init__(self, cache_dir: Path, split: str):
        data = torch.load(cache_dir / f'{split}.pt')
        self.feats  = data['feats']   # list of np.ndarray (F, T)
        self.labels = data['labels']  # list of int
        print(f'[{split}] loaded {len(self.feats)} chunks')
    def __len__(self): return len(self.feats)
    def __getitem__(self, idx):
        return torch.from_numpy(self.feats[idx]), torch.tensor(self.labels[idx], dtype=torch.float32)

def collate_fn(batch):
    feats, labels = zip(*batch)
    padded = pad_sequence([f.T for f in feats], batch_first=True).permute(0,2,1)
    return padded, torch.stack(labels)

# ── Model ─────────────────────────────────────────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, ic, oc, pool=(2,2)):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ic, oc, 3, padding=1, bias=False),
            nn.BatchNorm2d(oc), nn.ReLU(inplace=True), nn.MaxPool2d(pool))
    def forward(self, x): return self.net(x)

class AttentionPooling(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.a = nn.Linear(d, 1)
    def forward(self, x):
        return (torch.softmax(self.a(x), 1) * x).sum(1)

class SpectralModel(nn.Module):
    def __init__(self, n_feat=100, lstm_h=128, lstm_l=2):
        super().__init__()
        self.cnn  = nn.Sequential(ConvBlock(1,32), ConvBlock(32,64), ConvBlock(64,128))
        self.proj = nn.Linear(max(1, n_feat//8)*128, 128)
        self.lstm = nn.LSTM(128, lstm_h, lstm_l, batch_first=True, bidirectional=True)
        self.attn = AttentionPooling(lstm_h*2)
        self.head = nn.Sequential(nn.Linear(lstm_h*2, 64), nn.ReLU(), nn.Linear(64,1))
    def forward(self, x):
        B,F,T = x.shape
        x = self.cnn(x.unsqueeze(1))
        B2,C,F2,T2 = x.shape
        x = F.relu(self.proj(x.permute(0,3,1,2).reshape(B2,T2,C*F2)))
        x,_ = self.lstm(x)
        return torch.sigmoid(self.head(self.attn(x)).squeeze(-1))

# ── Metrics ───────────────────────────────────────────────────────────────────
def compute_eer(y, s):
    fpr, tpr, _ = roc_curve(y, s, pos_label=1)
    fnr = 1 - tpr
    try:    return brentq(lambda x: interp1d(fpr, fnr-fpr)(x), 0, 1)
    except: return float(np.mean(np.abs(fnr-fpr)))

def save_results(d, eer, auc, acc, f1):
    d = Path(d); d.mkdir(parents=True, exist_ok=True)
    (d/'results.txt').write_text(f'EER: {eer*100:.4f}%\nAUC: {auc:.4f}\nAccuracy: {acc*100:.4f}%\nF1: {f1:.4f}\n')
    (d/'results.json').write_text(json.dumps({'eer':eer,'auc':auc,'accuracy':acc,'f1':f1}, indent=4))
    print(f'Results saved → {d}')

# ── Training ──────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}  |  Batch: {BATCH_SIZE}  |  Epochs: {EPOCHS}')

train_ds = CachedSpectralDataset(CACHE_DIR, 'train')
val_ds   = CachedSpectralDataset(CACHE_DIR, 'val')
test_ds  = CachedSpectralDataset(CACHE_DIR, 'test')

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)

model     = SpectralModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.BCELoss()

best_eer, patience_ctr = float('inf'), 0
t_start = time.time()

for epoch in range(EPOCHS):
    # ── Train ──
    model.train()
    epoch_loss = 0.0
    for i, (feats, labels) in enumerate(train_loader):
        feats, labels = feats.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(feats), labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        elapsed = time.time() - t_start
        steps_done = epoch * len(train_loader) + i + 1
        steps_total = EPOCHS * len(train_loader)
        eta = elapsed / steps_done * (steps_total - steps_done)
        print(f'\r  Ep {epoch+1}/{EPOCHS} | Step {i+1}/{len(train_loader)} | Loss {loss.item():.4f} | ETA {int(eta//60)}m{int(eta%60)}s ', end='')
    scheduler.step()
    print(f'  avg_loss={epoch_loss/len(train_loader):.4f}')

    # ── Validate ──
    model.eval()
    val_labels, val_scores = [], []
    with torch.no_grad():
        for feats, labels in val_loader:
            val_scores += model(feats.to(device)).cpu().tolist()
            val_labels += labels.tolist()
    eer = compute_eer(np.array(val_labels), np.array(val_scores))
    print(f'  Val EER: {eer*100:.2f}%')

    if eer < best_eer:
        best_eer, patience_ctr = eer, 0
        torch.save({'model_state_dict': model.state_dict()}, RESULTS_DIR / 'best.pt')
        print('  ✔ New best model saved!')
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print('  Early stopping.'); break

# ── Test Evaluation ───────────────────────────────────────────────────────────
print('\nEvaluating on test set...')
model.load_state_dict(torch.load(RESULTS_DIR / 'best.pt')['model_state_dict'])
model.eval()
test_labels, test_scores = [], []
with torch.no_grad():
    for feats, labels in test_loader:
        test_scores += model(feats.to(device)).cpu().tolist()
        test_labels += labels.tolist()

tl, ts = np.array(test_labels), np.array(test_scores)
tp = (ts >= 0.5).astype(int)
fpr, tpr, _ = roc_curve(tl, ts, pos_label=1)
save_results(RESULTS_DIR, compute_eer(tl, ts), sklearn_auc(fpr, tpr),
             accuracy_score(tl, tp), f1_score(tl, tp))
print(f'[✔] Done. Best Val EER: {best_eer*100:.2f}%')
